# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [ ]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [ ]:
import pandas as pd

In [ ]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
98,All the info a man could possibly need to know...,good
736,Too many ads !,bad
312,Bought this for my daughter and she loves it.,good
325,GREAT price for a great magazine!!! If you lo...,good
731,"The quality of the articles has gone down, ful...",bad


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [ ]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [ ]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [ ]:
from sklearn import metrics

In [ ]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [ ]:
evaluate(sgd_clf)

Accuracy: 76.4%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

better_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

_ = better_clf.fit(train['review'], train['label'])
evaluate(better_clf)

Accuracy: 85.3%


I tried a different model without changing the dataset.
Multinomial Naive Bayes performed better than the baseline SGD classifier.
The accuracy improved from about 76% to about 85%.

## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [ ]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [ ]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [ ]:
train[train['review'].str.contains(r'<[^>]+>', regex=True, na=False)].head(10)

,review,label
0,Based on all the negative comments about Taste...,good
2,</tr>The magazine is not worth the cost of sub...,good
5,"The magazines are great, but I never received ...",good
10,"</div>It's not the fault of the magazine, I ju...",good
11,<li>dispatchEventBest magazine for current and...,bad
12,<li>onEmptiedBoth my husband and I really enjo...,bad
13,This magazine is filled with amazing recipes. ...,bad
17,</div>purchased for my wife who loves reading ...,bad
19,"<h4 class=""signature"">36 page pamphlet with LB...",good
22,No iPhone supportorg.python.core</FONT>,good


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [ ]:
import re

def is_bad_data(review: str) -> bool:
    return bool(re.search(r'<[^>]+>', review))

## Creating the cleaned training set

In [ ]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [ ]:
from sklearn import clone

In [ ]:
sgd_clf_clean = clone(sgd_clf)

In [ ]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [ ]:
evaluate(sgd_clf_clean)

Accuracy: 96.8%


## Part 1: Function Design (Core Task) - Task 1: Build a Model Comparison Function

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.base import clone

def compare_models(models: dict, X_train, y_train, X_test, y_test) -> pd.DataFrame:

    results = []

    for model_name, model_obj in models.items():

        current_model = clone(model_obj)

        current_model.fit(X_train, y_train)

        y_pred = current_model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, pos_label='good', zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label='good', zero_division=0)
        f1 = f1_score(y_test, y_pred, pos_label='good', zero_division=0)

        results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })

    return pd.DataFrame(results)


### Demonstrating the Model Comparison Function

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

print("--- Comparing models on the ORIGINAL data ---")

sgd_pipeline_original = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(random_state=42)),
])

mnb_pipeline_original = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

models_to_compare_original = {
    'SGDClassifier': sgd_pipeline_original,
    'Multinomial Naive Bayes': mnb_pipeline_original,
}

comparison_results_original = compare_models(
    models_to_compare_original,
    train['review'], train['label'],
    test['review'], test['label']
)

display(comparison_results_original.round(4))


print("\n--- Comparing models on the CLEANED data ---")

sgd_pipeline_cleaned = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(random_state=42)),
])

mnb_pipeline_cleaned = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

models_to_compare_cleaned = {
    'SGDClassifier (on clean data)': sgd_pipeline_cleaned,
    'Multinomial Naive Bayes (on clean data)': mnb_pipeline_cleaned,
}

comparison_results_cleaned = compare_models(
    models_to_compare_cleaned,
    train_clean['review'], train_clean['label'],
    test['review'], test['label']
)

display(comparison_results_cleaned.round(4))


--- Comparing models on the ORIGINAL data ---


,Model,Accuracy,Precision,Recall,F1-Score
0,SGDClassifier,0.763,0.7614,0.766,0.7637
1,Multinomial Naive Bayes,0.853,0.8537,0.852,0.8529



--- Comparing models on the CLEANED data ---


,Model,Accuracy,Precision,Recall,F1-Score
0,SGDClassifier (on clean data),0.969,0.9699,0.968,0.9690
1,Multinomial Naive Bayes (on clean data),0.956,0.9506,0.962,0.9563
